# Notebook 4 — Hydrogen Bond Analysis
## CURE Laboratory: General Chemistry II | HTLV-1 Protease Inhibitor Discovery

**When to complete:** Weeks 6–7  
**Learning objectives:**
- Apply the geometric definition of a hydrogen bond computationally
- Calculate H-bond occupancy (% of simulation time a bond is present)
- Explore how H-bond cutoff parameters affect results
- Create occupancy plots and interpret them in chemical terms
- Connect H-bond persistence to thermodynamic binding stability

**Prerequisites:** Notebooks 1–3 completed; MDAnalysis installed  
**Time estimate:** 90 minutes

---
> **Chemistry reminder:** A hydrogen bond requires:
> 1. A **donor** — an atom with a covalently bonded hydrogen (N–H, O–H)
> 2. An **acceptor** — an electronegative atom with lone pairs (N, O, F)
> 3. **Geometry:** D···A distance ≤ 3.5 Å AND D–H···A angle ≥ 120°
>
> H-bond strength depends on geometry, electronegativity, and competing interactions. In protein-ligand binding, H-bond occupancy (what fraction of simulation time the H-bond exists) is a proxy for its contribution to binding affinity.

---
## Part 1 — Setup


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import os, subprocess, sys

mpl.rcParams['figure.dpi'] = 120
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['axes.spines.top'] = False
mpl.rcParams['axes.spines.right'] = False

try:
    import MDAnalysis as mda
    from MDAnalysis.analysis.hydrogenbonds import HydrogenBondAnalysis
    MDA_AVAILABLE = True
    print(f"MDAnalysis {mda.__version__} available")
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'MDAnalysis', '--quiet'])
    try:
        import MDAnalysis as mda
        from MDAnalysis.analysis.hydrogenbonds import HydrogenBondAnalysis
        MDA_AVAILABLE = True
        print("MDAnalysis installed and loaded!")
    except ImportError:
        MDA_AVAILABLE = False
        print("Using pre-computed fallback data.")

TRAJ_AVAILABLE = os.path.exists("lysozyme_md.pdb") and os.path.exists("lysozyme_md.xtc")
print(f"Trajectory files available: {TRAJ_AVAILABLE}")


---
## Part 2 — The Geometry of Hydrogen Bonds

### 2.1 Implement the H-bond definition from scratch

Before using MDAnalysis's automated H-bond finder, you will implement the geometric definition yourself for a single frame. This ensures you understand what the algorithm is actually doing — and helps you interpret the results and troubleshoot when outputs seem unexpected.


In [ ]:
def angle_between_vectors(v1, v2):
    """
    Calculate the angle (in degrees) between two 3D vectors.
    Used to compute the D–H···A angle for H-bond detection.
    """
    cos_theta = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
    cos_theta = np.clip(cos_theta, -1.0, 1.0)   # avoid numerical errors
    return np.degrees(np.arccos(cos_theta))

def check_hbond(donor_pos, hydrogen_pos, acceptor_pos,
                dist_cutoff=3.5, angle_cutoff=120.0):
    """
    Check whether a D–H···A hydrogen bond exists based on:
    - Distance: donor-to-acceptor ≤ dist_cutoff (Å)
    - Angle: D–H···A ≥ angle_cutoff (degrees)

    Returns (is_hbond, distance, angle) tuple.
    """
    # Distance between donor (D) and acceptor (A)
    da_vector = acceptor_pos - donor_pos
    distance = np.linalg.norm(da_vector)

    # Angle D–H···A
    dh_vector = hydrogen_pos - donor_pos    # vector from D to H
    ha_vector = acceptor_pos - hydrogen_pos # vector from H to A
    angle = angle_between_vectors(dh_vector, ha_vector)

    is_hbond = (distance <= dist_cutoff) and (angle >= angle_cutoff)
    return is_hbond, round(distance, 3), round(angle, 1)

# Test with example coordinates representing a classic N–H···O=C backbone H-bond
# These coordinates are taken from a typical alpha helix geometry
donor_N    = np.array([0.0, 0.0, 0.0])      # backbone N (donor)
hydrogen_H = np.array([0.0, 1.01, 0.0])     # backbone H
acceptor_O = np.array([0.5, 3.8, 0.5])      # backbone O (acceptor)

is_hb, dist, ang = check_hbond(donor_N, hydrogen_H, acceptor_O)
print(f"H-bond test 1 (typical alpha helix backbone H-bond):")
print(f"  D···A distance: {dist:.2f} Å  (cutoff ≤ 3.5 Å)")
print(f"  D–H···A angle:  {ang:.1f}°    (cutoff ≥ 120°)")
print(f"  H-bond detected: {is_hbond}")

# Test with a geometry that should NOT be an H-bond (too far)
acceptor_far = np.array([0.0, 5.0, 0.0])
is_hb2, dist2, ang2 = check_hbond(donor_N, hydrogen_H, acceptor_far)
print(f"\nH-bond test 2 (too far apart):")
print(f"  D···A distance: {dist2:.2f} Å  → {'✓' if dist2 <= 3.5 else '✗ FAILS distance cutoff'}")
print(f"  H-bond detected: {is_hb2}")

# Test with a geometry that fails angle criterion (bent H-bond)
acceptor_bent = np.array([0.0, 2.5, 3.5])
is_hb3, dist3, ang3 = check_hbond(donor_N, hydrogen_H, acceptor_bent)
print(f"\nH-bond test 3 (bent — poor geometry):")
print(f"  D–H···A angle: {ang3:.1f}° → {'✓' if ang3 >= 120 else '✗ FAILS angle cutoff'}")
print(f"  H-bond detected: {is_hb3}")


### ✏️ ANNOTATION REQUIRED

1. In Test 1 (alpha helix backbone H-bond), was the H-bond detected? Does this make chemical sense? (Alpha helices are stabilized by backbone H-bonds between residue *i* and residue *i+4*.)
2. Which cutoff (distance or angle) is typically more restrictive in practice? Look at the example values and think about when each criterion would eliminate an interaction.
3. **Design question:** If you changed the distance cutoff from 3.5 Å to 3.0 Å, would you detect *more* or *fewer* H-bonds? If you changed the angle cutoff from 120° to 150°, would you detect *more* or *fewer*? How would each change affect the biological interpretation of your results?


### My Annotation for Section 2:

*[YOUR ANSWER HERE]*


---
## Part 3 — How Cutoff Parameters Affect H-Bond Counts

### 3.1 Explore cutoff sensitivity

One of the most important skills in computational analysis is understanding how your *parameters* affect your *results*. Before running your Phase 2 analysis, you should know how sensitive the H-bond occupancy values are to the distance and angle cutoffs you choose.

The cell below uses pre-computed H-bond count data to illustrate this effect. When you run your Phase 2 analysis, you will use the standard cutoffs (3.5 Å, 120°) unless you have a scientific reason to deviate.


In [ ]:
# Demonstrate cutoff sensitivity using simulated H-bond count data
# This models what you would see varying cutoffs on a real system

np.random.seed(42)
n_frames = 200
time_arr = np.linspace(0, 5, n_frames)

# Simulate H-bond counts under different cutoff conditions
# More permissive cutoffs = more H-bonds detected
def simulate_hbond_count(base_count, noise_level=0.8, seed=0):
    np.random.seed(seed)
    signal = base_count + np.random.normal(0, noise_level, n_frames)
    return np.clip(signal.astype(int), 0, base_count + 5)

cutoff_scenarios = {
    'Strict (3.0 Å, 150°)':   simulate_hbond_count(2, seed=1),
    'Standard (3.5 Å, 120°)': simulate_hbond_count(4, seed=2),
    'Permissive (4.0 Å, 100°)': simulate_hbond_count(7, seed=3),
}

fig, axes = plt.subplots(2, 1, figsize=(11, 7))

colors = ['#CC3300', '#2E5EAA', '#27AE60']
for (label, counts), color in zip(cutoff_scenarios.items(), colors):
    axes[0].plot(time_arr, counts, color=color, linewidth=1.5, alpha=0.8, label=label)

axes[0].set_xlabel('Simulation Time (ns)', fontsize=11)
axes[0].set_ylabel('H-Bonds Detected per Frame', fontsize=11)
axes[0].set_title('Effect of Cutoff Parameters on H-Bond Detection Count', fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.25)

# Show the occupancy difference
labels = list(cutoff_scenarios.keys())
mean_counts = [np.mean(v) for v in cutoff_scenarios.values()]
std_counts = [np.std(v) for v in cutoff_scenarios.values()]
axes[1].bar(range(len(labels)), mean_counts, color=colors, alpha=0.8,
            yerr=std_counts, capsize=6, edgecolor='white')
axes[1].set_xticks(range(len(labels)))
axes[1].set_xticklabels(labels, fontsize=10)
axes[1].set_ylabel('Mean H-Bonds per Frame', fontsize=11)
axes[1].set_title('Mean H-Bond Count by Cutoff Setting', fontweight='bold')
axes[1].grid(True, axis='y', alpha=0.25)

plt.tight_layout()
plt.savefig('hbond_cutoff_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

print("Cutoff sensitivity summary:")
for label, counts in cutoff_scenarios.items():
    print(f"  {label}: mean = {np.mean(counts):.2f} ± {np.std(counts):.2f}")


---
## Part 4 — Calculating H-Bond Occupancy

### 4.1 The occupancy concept

For each potential H-bond between a specific donor-acceptor pair, occupancy is:

**Occupancy (%) = (frames where H-bond exists / total production frames) × 100**

An 85% occupancy means the H-bond is present in 85 out of every 100 simulation frames. This is a strong, persistent interaction. A 15% occupancy is transient — present occasionally but not a dominant interaction.

**Why this matters for your research:** When comparing inhibitor classes, a statine-based inhibitor that forms an H-bond with 80% occupancy to a loop residue is interacting more persistently than an HEA inhibitor with 30% occupancy to the same residue. This difference may explain differences in inhibitor effectiveness.


In [ ]:
if TRAJ_AVAILABLE and MDA_AVAILABLE:
    u = mda.Universe("lysozyme_md.pdb", "lysozyme_md.xtc")
    production_start = int(u.trajectory.n_frames * 0.2)

    # Run hydrogen bond analysis for the production phase
    hba = HydrogenBondAnalysis(
        universe=u,
        donors_sel=None,   # auto-detect all N–H and O–H donors
        acceptors_sel=None, # auto-detect all N, O acceptors
        d_a_cutoff=3.5,
        d_h_a_angle_cutoff=120.0,
        update_selections=False
    )
    print("Running H-bond analysis (this may take a few minutes)...")
    hba.run(start=production_start)

    results_df = hba.results.hbonds
    print(f"Total H-bond events detected: {len(results_df)}")
    HB_SOURCE = "live"
else:
    # 🔁 Pre-computed occupancy data — represents typical backbone H-bonds in lysozyme
    print("Using pre-computed H-bond occupancy data for lysozyme.")
    np.random.seed(99)
    # Model 20 H-bond pairs with varying occupancies
    n_pairs = 20
    donor_residues   = np.random.choice(range(1, 129), n_pairs, replace=False)
    acceptor_residues = donor_residues + np.random.choice([-4, -3, 3, 4], n_pairs)
    acceptor_residues = np.clip(acceptor_residues, 1, 129)

    # Realistic distribution: most backbone H-bonds >70% (in helices), loops lower
    occupancies = np.concatenate([
        np.random.uniform(70, 98, 12),   # stable helix H-bonds
        np.random.uniform(20, 65, 5),    # partial H-bonds
        np.random.uniform(5, 20, 3)      # transient H-bonds
    ])

    results_df = pd.DataFrame({
        'donor_resid': donor_residues,
        'acceptor_resid': acceptor_residues,
        'occupancy': occupancies
    }).sort_values('occupancy', ascending=False).reset_index(drop=True)

    HB_SOURCE = "precomputed"
    print(f"Pre-computed data: {len(results_df)} H-bond pairs")


In [ ]:
# Calculate and display occupancy for each H-bond pair
if HB_SOURCE == "live":
    # Calculate occupancy from raw MDAnalysis results
    total_frames = len(u.trajectory) - production_start
    pair_counts = {}
    for row in results_df:
        key = (int(row[1]), int(row[3]))  # (donor_resid, acceptor_resid)
        pair_counts[key] = pair_counts.get(key, 0) + 1

    occupancy_data = [
        {'donor': k[0], 'acceptor': k[1],
         'count': v, 'occupancy': round(v / total_frames * 100, 1)}
        for k, v in pair_counts.items()
    ]
    occ_df = pd.DataFrame(occupancy_data).sort_values('occupancy', ascending=False)
else:
    occ_df = results_df.copy()
    occ_df.columns = ['donor', 'acceptor', 'occupancy']

print("Top 15 H-bonds by occupancy:")
print(occ_df.head(15).to_string(index=False))

# Categorize by occupancy strength
strong   = (occ_df['occupancy'] > 70).sum()
moderate = ((occ_df['occupancy'] > 30) & (occ_df['occupancy'] <= 70)).sum()
weak     = (occ_df['occupancy'] <= 30).sum()
print(f"\nH-bond strength categories:")
print(f"  Strong (>70%):      {strong}")
print(f"  Moderate (30–70%):  {moderate}")
print(f"  Weak (<30%):        {weak}")


In [ ]:
# Visualize H-bond occupancy distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Histogram of all occupancy values
ax1 = axes[0]
ax1.hist(occ_df['occupancy'], bins=20, color='#2E5EAA', alpha=0.8, edgecolor='white')
ax1.axvline(70, color='#27AE60', linestyle='--', linewidth=2, label='Strong (>70%)')
ax1.axvline(30, color='#CC3300', linestyle='--', linewidth=2, label='Weak threshold (30%)')
ax1.set_xlabel('H-Bond Occupancy (%)', fontsize=12)
ax1.set_ylabel('Number of H-Bond Pairs', fontsize=12)
ax1.set_title('Distribution of H-Bond Occupancies', fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.25)

# Horizontal bar chart of top H-bonds
ax2 = axes[1]
top_n = min(15, len(occ_df))
top_hb = occ_df.head(top_n)
y_labels = [f"Res {int(row.donor)} → Res {int(row.acceptor)}"
            for _, row in top_hb.iterrows()]
bar_colors = ['#27AE60' if o > 70 else ('#E8A838' if o > 30 else '#CC3300')
              for o in top_hb['occupancy']]

bars = ax2.barh(range(top_n), top_hb['occupancy'], color=bar_colors, alpha=0.85)
ax2.set_yticks(range(top_n))
ax2.set_yticklabels(y_labels, fontsize=9)
ax2.set_xlabel('Occupancy (%)', fontsize=11)
ax2.set_title(f'Top {top_n} H-Bond Pairs by Occupancy', fontweight='bold')
ax2.axvline(70, color='#27AE60', linestyle='--', alpha=0.7)
ax2.axvline(30, color='#CC3300', linestyle='--', alpha=0.7)
ax2.set_xlim(0, 105)
ax2.grid(True, axis='x', alpha=0.25)

plt.tight_layout()
plt.savefig('hbond_occupancy.png', dpi=150, bbox_inches='tight')
plt.show()

if HB_SOURCE == "precomputed":
    print("[Pre-computed data used — values are representative of lysozyme backbone H-bonds]")


### ✏️ ANNOTATION REQUIRED

1. **Interpret the distribution:** What fraction of all detected H-bonds are "strong" (>70% occupancy)? What does it mean chemically for most backbone H-bonds to be persistent?
2. **Equilibrium perspective:** A 70% occupancy H-bond corresponds to what equilibrium constant K? (Hint: use K = [bonded]/[unbonded] = 0.70/0.30, then ΔG = −RT ln K at T = 300 K.) Is this a thermodynamically favorable interaction?
3. **Connection to your research:** In Phase 2, you will calculate H-bond occupancy between your inhibitor and the HTLV-1 protease (especially loop residues 91–100). If the most important H-bond between your inhibitor and the protease has 85% occupancy, what would that tell you about binding stability? How would you report this in your manuscript?
4. **Parameter choice:** You used the "standard" cutoffs (3.5 Å, 120°) in this analysis. Your instructor chose these values — but why are they standard? (Hint: consider what the typical N···O distance is in a crystal structure H-bond, and what angle makes a "good" H-bond geometrically.)


### My Annotation for Section 4:

*[YOUR ANSWER HERE — address all four questions, minimum 6 sentences total]*


---
## ✅ Notebook 4 Completion Checklist

- [ ] H-bond geometry functions implemented and tested (all 3 test cases run)
- [ ] Cutoff sensitivity plot generated
- [ ] Occupancy distribution and bar chart generated
- [ ] All **ANNOTATION REQUIRED** sections completed
- [ ] ΔG calculation completed in annotation (Section 4, question 2)
- [ ] Notebook saved and submitted

---
## 🔗 You are now ready for Phase 2

You have learned:
- ✓ RMSD — how much the protein moves globally (Notebook 3)
- ✓ RMSF — which residues are most flexible (Notebook 3)
- ✓ H-bond detection — the geometric criteria (Notebook 4)
- ✓ H-bond occupancy — how persistent an interaction is (Notebook 4)

**Notebook 5** will apply all of these tools to the HTLV-1 protease with your assigned inhibitor. You will only need to change the input file names, residue numbers for the polyproline loop, and the inhibitor atom names. All the code is already written.
